# Alert Explanation — LLM + RAG

Google Colab prototype for explaining WealthWise financial alerts using retrieved user context.

**Flow:** Alert → retrieve relevant financial context → prompt → Ollama LLM → clear explanation + suggested action.

In [1]:
!pip -q install chromadb sentence-transformers pandas requests


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 38.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 16.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 26.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.2/19.2 MB 44.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2/137.2 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.6/204.6 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 94.7/94.7 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 2.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently ta

In [2]:
import requests
import pandas as pd
import chromadb
from sentence_transformers import SentenceTransformer


## 1. Example WealthWise data

Replace these examples with data from the Django backend.

In [3]:
transactions = pd.DataFrame([
    ['2026-08-10', 'Swiggy', 'Food', 1250],
    ['2026-08-12', 'Amazon', 'Shopping', 4200],
    ['2026-08-14', 'Uber', 'Transport', 900],
], columns=['date','merchant','category','amount'])

budgets = pd.DataFrame([
    ['Food', 5000, 4750],
    ['Shopping', 4000, 4200],
    ['Transport', 3000, 900],
], columns=['category','budget','spent'])

goals = pd.DataFrame([
    ['Emergency Fund', 100000, 35000],
], columns=['goal','target','saved'])

alert = {
    'type': 'Budget Exceeded',
    'category': 'Shopping',
    'message': 'Shopping spending has exceeded the monthly budget.',
    'severity': 'HIGH'
}

display(transactions)
display(budgets)
display(goals)
print(alert)


,date,merchant,category,amount
0,2026-08-10,Swiggy,Food,1250
1,2026-08-12,Amazon,Shopping,4200
2,2026-08-14,Uber,Transport,900


,category,budget,spent
0,Food,5000,4750
1,Shopping,4000,4200
2,Transport,3000,900


,goal,target,saved
0,Emergency Fund,100000,35000


{'type': 'Budget Exceeded', 'category': 'Shopping', 'message': 'Shopping spending has exceeded the monthly budget.', 'severity': 'HIGH'}


## 2. Build RAG knowledge base

In [4]:
documents = []

for _, r in transactions.iterrows():
    documents.append(f"Transaction on {r.date}: {r.merchant}, category {r.category}, amount INR {r.amount:.2f}.")

for _, r in budgets.iterrows():
    remaining = r.budget - r.spent
    documents.append(f"Budget for {r.category}: INR {r.budget:.2f}; spent INR {r.spent:.2f}; remaining INR {remaining:.2f}.")

for _, r in goals.iterrows():
    documents.append(f"Goal {r.goal}: target INR {r.target:.2f}; saved INR {r.saved:.2f}; remaining INR {r.target-r.saved:.2f}.")

embedding_model = SentenceTransformer('all-MiniLM-L6-v2')
embeddings = embedding_model.encode(documents, normalize_embeddings=True).tolist()

client = chromadb.Client()
collection = client.get_or_create_collection('wealthwise_alert_context')
collection.upsert(ids=[f'doc_{i}' for i in range(len(documents))], documents=documents, embeddings=embeddings)
print('RAG documents:', collection.count())


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

RAG documents: 7


## 3. Retrieve alert-specific context

In [5]:
def retrieve_context(alert, top_k=6):
    query = f"Explain this financial alert: {alert['type']} {alert['category']} {alert['message']}"
    q = embedding_model.encode([query], normalize_embeddings=True).tolist()
    result = collection.query(query_embeddings=q, n_results=top_k)
    return result['documents'][0]

context = retrieve_context(alert)
for item in context:
    print('-', item)


- Budget for Shopping: INR 4000.00; spent INR 4200.00; remaining INR -200.00.
- Budget for Transport: INR 3000.00; spent INR 900.00; remaining INR 2100.00.
- Budget for Food: INR 5000.00; spent INR 4750.00; remaining INR 250.00.
- Transaction on 2026-08-12: Amazon, category Shopping, amount INR 4200.00.
- Goal Emergency Fund: target INR 100000.00; saved INR 35000.00; remaining INR 65000.00.
- Transaction on 2026-08-14: Uber, category Transport, amount INR 900.00.


## 4. Build alert explanation prompt

In [6]:
def build_prompt(alert, context):
    return f'''You are WealthWise, a financial alert explanation assistant.

Explain the alert in simple language using ONLY the alert and retrieved financial context.
Do not invent transactions, amounts, causes, or financial facts.
Explain why the alert was triggered using exact values when available.
Then provide 2 practical actions the user can consider.
Do not give unsupported financial advice.

Return exactly these sections:
Why this alert appeared
What it means
What you can do

ALERT:
{alert}

RETRIEVED FINANCIAL CONTEXT:
{chr(10).join('- '+x for x in context)}
'''

prompt = build_prompt(alert, context)
print(prompt)


You are WealthWise, a financial alert explanation assistant.

Explain the alert in simple language using ONLY the alert and retrieved financial context.
Do not invent transactions, amounts, causes, or financial facts.
Explain why the alert was triggered using exact values when available.
Then provide 2 practical actions the user can consider.
Do not give unsupported financial advice.

Return exactly these sections:
Why this alert appeared
What it means
What you can do

ALERT:
{'type': 'Budget Exceeded', 'category': 'Shopping', 'message': 'Shopping spending has exceeded the monthly budget.', 'severity': 'HIGH'}

RETRIEVED FINANCIAL CONTEXT:
- Budget for Shopping: INR 4000.00; spent INR 4200.00; remaining INR -200.00.
- Budget for Transport: INR 3000.00; spent INR 900.00; remaining INR 2100.00.
- Budget for Food: INR 5000.00; spent INR 4750.00; remaining INR 250.00.
- Transaction on 2026-08-12: Amazon, category Shopping, amount INR 4200.00.
- Goal Emergency Fund: target INR 100000.00; sa

## 5. Generate explanation with Ollama

For local execution use `http://localhost:11434`. For Google Colab, replace it with your Cloudflare Tunnel URL.

In [7]:
OLLAMA_URL = 'https://bikini-chosen-bye-collective.trycloudflare.com/api/generate'
OLLAMA_MODEL = 'llama3.2'

def generate_alert_explanation(prompt):
    response = requests.post(
        OLLAMA_URL,
        json={'model': OLLAMA_MODEL, 'prompt': prompt, 'stream': False},
        timeout=120
    )
    response.raise_for_status()
    return response.json()['response']

try:
    explanation = generate_alert_explanation(prompt)
    print(explanation)
except Exception as e:
    print('Ollama unavailable:', e)


**Why this alert appeared**

This alert was triggered because the monthly budget for Shopping has been exceeded by spending INR 4200.00, which is more than the allocated budget of INR 4000.00.

**What it means**

The user's shopping expenses have surpassed their budget, leaving a deficit of INR 200.00 in the remaining balance. This indicates that the user needs to be more mindful of their spending habits in this category.

**What you can do**

1. Review and adjust your shopping list to identify areas where you can cut back on unnecessary purchases.
2. Consider implementing a temporary budget freeze for non-essential items until the end-of-month budget is adjusted to reflect the actual expenditure.


## 6. Deterministic fallback

This allows the prototype to work even when Ollama is unavailable.

In [8]:
row = budgets[budgets['category'].eq(alert['category'])].iloc[0]
difference = row['spent'] - row['budget']
fallback = f'''## Why this alert appeared
Your {alert['category']} spending is INR {row['spent']:,.0f}, which is INR {difference:,.0f} above the INR {row['budget']:,.0f} budget.

## What it means
The {alert['category']} budget has been exceeded based on the available WealthWise data.

## What you can do
1. Review recent {alert['category']} transactions and identify discretionary spending.
2. Monitor remaining spending in other budget categories before making additional discretionary purchases.'''

if 'explanation' not in globals():
    print(fallback)


## 7. Export explanation

In [9]:
final_explanation = explanation if 'explanation' in globals() else fallback
with open('wealthwise_alert_explanation.md', 'w', encoding='utf-8') as f:
    f.write(final_explanation)
print('Saved: wealthwise_alert_explanation.md')


Saved: wealthwise_alert_explanation.md


## WealthWise integration

The alert engine should first create a structured alert from deterministic rules or ML models. Django then sends the alert to this RAG layer. ChromaDB retrieves relevant transactions, budgets, goals, forecasts, and other context. The LLM explains the already-determined alert; it should **not decide whether an alert is true**. This keeps the system grounded and prevents the LLM from inventing alerts.